# Análisis Completo del Modelo LSTM para Predicción de MERVAL

Este notebook combina:
- Explicación e interpretación del modelo
- Comparación con y sin features de sentimiento
- Visualizaciones de resultados
- Análisis de hiperparámetros
- Interpretación de probabilidades y predicciones

## Diferencia entre Probabilidad y Accuracy

- **Probabilidad**: Confianza del modelo (0-1). Ej: 0.75 = 75% de confianza de que MERVAL subirá
- **Accuracy**: % de predicciones correctas. Ej: 0.65 = 65% de aciertos

La probabilidad se calcula aplicando sigmoid al logit del modelo. Si probabilidad ≥ 0.5, predice "subirá", si no "bajará".


## 1. Configuración e Importaciones


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Agregar src al path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

# Importar módulos del proyecto
from src.modelo.model_utils import (
    MaskedSeqDataset,
    LSTMBinary,
    rolling_splits,
    train_fold
)
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)

# Configuración de visualización
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
%matplotlib inline

print("Modulos importados correctamente")


Modulos importados correctamente


## 2. Explicación del Modelo

### ¿Qué hace el modelo?

El modelo es una **LSTM (Long Short-Term Memory)** que predice si el índice MERVAL **subirá o bajará** al día siguiente (clasificación binaria).

### Arquitectura:
```
Input: Secuencia de N días × M features
    ↓
LSTM Layer (captura patrones temporales)
    ↓
Dropout (regularización)
    ↓
Fully Connected Layer
    ↓
Output: Logit → Sigmoid → Probabilidad (0-1)
```

### Interpretación:
- Si probabilidad ≥ 0.5 → Predice "subirá" (clase 1)
- Si probabilidad < 0.5 → Predice "bajará" (clase 0)
- Accuracy mide qué tan bien predice en general


## 3. Funciones Auxiliares para Entrenamiento con Probabilidades


In [2]:
def train_model_with_predictions(
    df: pd.DataFrame,
    seq_len: int,
    hidden_size: int,
    dropout: float,
    lr: float,
    epochs: int,
    batch_size: int,
    device: str = 'cpu',
    return_predictions: bool = True
) -> dict:
    """
    Entrena modelo y retorna métricas + predicciones + probabilidades.
    """
    device = torch.device(device)
    
    # Preparar datos
    feature_cols = [c for c in df.columns if c != 'prediccion']
    y_raw = df['prediccion'].astype('float32').to_numpy()
    X_raw = df[feature_cols].to_numpy(dtype='float32')
    
    # Split simple (80% train, 20% test)
    split_idx = int(len(X_raw) * 0.8)
    X_train_raw = X_raw[:split_idx]
    y_train = y_raw[:split_idx]
    X_test_raw = X_raw[split_idx:]
    y_test = y_raw[split_idx:]
    
    # Normalizar
    X_mean = X_train_raw.mean(axis=0, keepdims=True)
    X_std = X_train_raw.std(axis=0, keepdims=True)
    X_train = (X_train_raw - X_mean) / (X_std + 1e-8)
    X_test = (X_test_raw - X_mean) / (X_std + 1e-8)
    
    # Crear datasets
    train_ds = MaskedSeqDataset(X_train, y_train, seq_len)
    test_ds = MaskedSeqDataset(X_test, y_test, seq_len)
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    
    # Crear modelo
    model = LSTMBinary(
        input_size=len(feature_cols),
        hidden_size=hidden_size,
        num_layers=1,
        dropout=dropout,
        bidirectional=False
    )
    model.to(device)
    
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Entrenar
    train_losses = []
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_losses = []
        
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())
        
        avg_loss = np.mean(epoch_losses)
        train_losses.append(avg_loss)
        if epoch % 5 == 0:
            print(f"Epoch {epoch}/{epochs} - Loss: {avg_loss:.4f}")
    
    # Evaluar
    model.eval()
    test_probs = []
    test_labels = []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            logits = model(xb)
            probs = torch.sigmoid(logits).cpu().numpy()
            test_probs.append(probs)
            test_labels.append(yb.numpy())
    
    test_probs = np.concatenate(test_probs)
    test_labels = np.concatenate(test_labels)
    test_preds = (test_probs > 0.5).astype(int)
    
    # Calcular métricas
    accuracy = accuracy_score(test_labels, test_preds)
    precision = precision_score(test_labels, test_preds, zero_division=0)
    recall = recall_score(test_labels, test_preds, zero_division=0)
    f1 = f1_score(test_labels, test_preds, zero_division=0)
    
    result = {
        'metrics': {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
        },
        'train_losses': train_losses,
    }
    
    if return_predictions:
        result.update({
            'y_true': test_labels,
            'y_pred': test_preds,
            'probabilities': test_probs,
        })
    
    return result

def create_dataset_without_sentiment(csv_path: str) -> str:
    """Crea versión del dataset sin features de sentimiento."""
    df = pd.read_csv(csv_path, sep=';')
    
    # Mantener solo retorno_log_merval, retorno_log_dolar y prediccion
    cols_to_keep = ['retorno_log_merval', 'retorno_log_dolar', 'prediccion']
    df_no_sentiment = df[cols_to_keep].copy()
    
    output_path = csv_path.replace('.csv', '_sin_sentiment.csv')
    df_no_sentiment.to_csv(output_path, index=False, sep=';')
    
    print(f"Dataset sin sentimiento creado: {output_path}")
    print(f"Features eliminadas: {len(df.columns) - len(cols_to_keep)}")
    print(f"Features restantes: {len(cols_to_keep) - 1}")
    
    return output_path

print("Funciones auxiliares definidas")


Funciones auxiliares definidas


## 4. Cargar Datos


In [3]:
# Cargar dataset de entrenamiento
CSV_PATH = "src/modelo/data/data_train.csv"
csv_path = Path(CSV_PATH)

if csv_path.exists():
    df = pd.read_csv(CSV_PATH, sep=';')
    print(f"Dataset cargado: {len(df)} filas, {len(df.columns)} columnas")
    print(f"\nColumnas:")
    for col in df.columns:
        print(f"  - {col}")
    
    print(f"\nPrimeras filas:")
    display(df.head())
    
    print(f"\nEstadísticas básicas:")
    display(df.describe())
    
    # Rellenar NaNs en features con 0
    feature_cols = [c for c in df.columns if c != 'prediccion']
    df[feature_cols] = df[feature_cols].fillna(0)
else:
    print(f"No se encontro el archivo {CSV_PATH}")
    print("Ejecuta primero el notebook generar_datasets.ipynb para crear el dataset")


No se encontro el archivo src/modelo/data/data_train.csv
Ejecuta primero el notebook generar_datasets.ipynb para crear el dataset


## 5. Configuración del Modelo


In [4]:
# Configuración del modelo
SEQ_LEN = 90  # Ventana temporal (días históricos)
HIDDEN_SIZE = 28  # Neuronas en la capa LSTM
EPOCHS = 40  # Iteraciones de entrenamiento
BATCH_SIZE = 16  # Tamaño del batch
LEARNING_RATE = 0.00057  # Tasa de aprendizaje
DROPOUT = 0.46  # Regularización
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print("Configuracion del modelo:")
print(f"   Lookback: {SEQ_LEN} dias")
print(f"   Hidden Size: {HIDDEN_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Learning Rate: {LEARNING_RATE}")
print(f"   Device: {DEVICE}")


Configuracion del modelo:
   Lookback: 90 dias
   Hidden Size: 28
   Epochs: 40
   Batch Size: 16
   Learning Rate: 0.00057
   Device: cuda


## 6. Entrenar Modelo CON Sentimiento


In [5]:
print("Entrenando modelo CON features de sentimiento...")
print("="*60)

feature_cols_with = [c for c in df.columns if c != 'prediccion']
print(f"\nFeatures utilizadas ({len(feature_cols_with)}):")
for i, col in enumerate(feature_cols_with, 1):
    print(f"  {i}. {col}")

results_with = train_model_with_predictions(
    df,
    seq_len=SEQ_LEN,
    hidden_size=HIDDEN_SIZE,
    dropout=DROPOUT,
    lr=LEARNING_RATE,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    return_predictions=True
)

print("\nEntrenamiento completado")
print("\nMetricas:")
for metric, value in results_with['metrics'].items():
    print(f"   {metric.capitalize()}: {value:.4f}")

# Mostrar algunas probabilidades
print("\nEjemplo de Probabilidades (primeras 10 predicciones):")
df_probs_example = pd.DataFrame({
    'y_true': results_with['y_true'][:10],
    'y_pred': results_with['y_pred'][:10],
    'probabilidad': results_with['probabilities'][:10]
})
df_probs_example['interpretacion'] = df_probs_example['probabilidad'].apply(
    lambda p: f"{'Subira' if p >= 0.5 else 'Bajara'} ({p:.1%} confianza)"
)
display(df_probs_example)


Entrenando modelo CON features de sentimiento...


NameError: name 'df' is not defined

## 7. Entrenar Modelo SIN Sentimiento (Baseline)


In [ ]:
# Crear dataset sin sentimiento
csv_no_sentiment = create_dataset_without_sentiment(CSV_PATH)
df_no_sentiment = pd.read_csv(csv_no_sentiment, sep=';')
feature_cols_no_sentiment = [c for c in df_no_sentiment.columns if c != 'prediccion']
df_no_sentiment[feature_cols_no_sentiment] = df_no_sentiment[feature_cols_no_sentiment].fillna(0)

print("Entrenando modelo SIN features de sentimiento (baseline)...")
print("="*60)

print(f"\nFeatures utilizadas ({len(feature_cols_no_sentiment)}):")
for i, col in enumerate(feature_cols_no_sentiment, 1):
    print(f"  {i}. {col}")

results_without = train_model_with_predictions(
    df_no_sentiment,
    seq_len=SEQ_LEN,
    hidden_size=HIDDEN_SIZE,
    dropout=DROPOUT,
    lr=LEARNING_RATE,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    return_predictions=True
)

print("\nEntrenamiento completado")
print("\nMetricas:")
for metric, value in results_without['metrics'].items():
    print(f"   {metric.capitalize()}: {value:.4f}")


## 8. Comparación de Modelos


In [ ]:
# Crear DataFrame de comparación
comparison_data = {
    'Metrica': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Con Sentimiento': [
        results_with['metrics']['accuracy'],
        results_with['metrics']['precision'],
        results_with['metrics']['recall'],
        results_with['metrics']['f1']
    ],
    'Sin Sentimiento': [
        results_without['metrics']['accuracy'],
        results_without['metrics']['precision'],
        results_without['metrics']['recall'],
        results_without['metrics']['f1']
    ]
}

df_comparison = pd.DataFrame(comparison_data)
df_comparison['Mejora'] = df_comparison['Con Sentimiento'] - df_comparison['Sin Sentimiento']
df_comparison['Mejora %'] = (df_comparison['Mejora'] / df_comparison['Sin Sentimiento'] * 100).round(2)

print("COMPARACION DE MODELOS")
print("="*60)
display(df_comparison)

# Visualización
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(df_comparison))
width = 0.35

bars1 = ax.bar(x - width/2, df_comparison['Con Sentimiento'], width, 
               label='Con Sentimiento', alpha=0.8, color='#3498db')
bars2 = ax.bar(x + width/2, df_comparison['Sin Sentimiento'], width,
               label='Sin Sentimiento', alpha=0.8, color='#e74c3c')

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Comparacion de Metricas: Con vs Sin Sentimiento', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(df_comparison['Metrica'])
ax.legend()
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3, axis='y')

# Agregar valores en las barras
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.3f}',
               ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

# Resumen de mejora
print("\nRESUMEN DE MEJORA")
print("="*60)
for _, row in df_comparison.iterrows():
    sign = "+" if row['Mejora'] > 0 else ""
    print(f"{row['Metrica']}: {sign}{row['Mejora']:.4f} ({sign}{row['Mejora %']:.2f}%)")


In [ ]:
# Crear DataFrame con probabilidades y predicciones
df_probs_with = pd.DataFrame({
    'y_true': results_with['y_true'],
    'y_pred': results_with['y_pred'],
    'probabilidad': results_with['probabilities'],
    'correcto': results_with['y_true'] == results_with['y_pred']
})

df_probs_without = pd.DataFrame({
    'y_true': results_without['y_true'],
    'y_pred': results_without['y_pred'],
    'probabilidad': results_without['probabilities'],
    'correcto': results_without['y_true'] == results_without['y_pred']
})

print("PROBABILIDADES DEL MODELO")
print("="*60)
print("\nPrimeras 20 predicciones con probabilidades (CON Sentimiento):")
display(df_probs_with.head(20))

print("\nDISTRIBUCION DE PROBABILIDADES")
print("="*60)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histograma de probabilidades - Con sentimiento
axes[0, 0].hist(df_probs_with['probabilidad'], bins=30, alpha=0.7, color='#3498db', edgecolor='black')
axes[0, 0].axvline(0.5, color='red', linestyle='--', linewidth=2, label='Umbral (0.5)')
axes[0, 0].set_title('Distribución de Probabilidades (Con Sentimiento)', fontweight='bold')
axes[0, 0].set_xlabel('Probabilidad')
axes[0, 0].set_ylabel('Frecuencia')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Histograma de probabilidades - Sin sentimiento
axes[0, 1].hist(df_probs_without['probabilidad'], bins=30, alpha=0.7, color='#e74c3c', edgecolor='black')
axes[0, 1].axvline(0.5, color='red', linestyle='--', linewidth=2, label='Umbral (0.5)')
axes[0, 1].set_title('Distribución de Probabilidades (Sin Sentimiento)', fontweight='bold')
axes[0, 1].set_xlabel('Probabilidad')
axes[0, 1].set_ylabel('Frecuencia')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Box plot por clase real - Con sentimiento
df_probs_with.boxplot(column='probabilidad', by='y_true', ax=axes[1, 0])
axes[1, 0].set_title('Probabilidad por Clase Real (Con Sentimiento)', fontweight='bold')
axes[1, 0].set_xlabel('Clase Real (0=Baja, 1=Sube)')
axes[1, 0].set_ylabel('Probabilidad')
axes[1, 0].set_ylim([0, 1])

# Box plot por clase real - Sin sentimiento
df_probs_without.boxplot(column='probabilidad', by='y_true', ax=axes[1, 1])
axes[1, 1].set_title('Probabilidad por Clase Real (Sin Sentimiento)', fontweight='bold')
axes[1, 1].set_xlabel('Clase Real (0=Baja, 1=Sube)')
axes[1, 1].set_ylabel('Probabilidad')
axes[1, 1].set_ylim([0, 1])

plt.tight_layout()
plt.show()

# Estadísticas de probabilidades
print("\nESTADISTICAS DE PROBABILIDADES")
print("="*60)
print("\nCon Sentimiento:")
print(df_probs_with['probabilidad'].describe())
print("\nSin Sentimiento:")
print(df_probs_without['probabilidad'].describe())

# Análisis de confianza
print("\nANALISIS DE CONFIANZA")
print("="*60)
for threshold in [0.6, 0.7, 0.8, 0.9]:
    high_conf_with = ((df_probs_with['probabilidad'] >= threshold) | 
                      (df_probs_with['probabilidad'] <= 1-threshold))
    if high_conf_with.sum() > 0:
        acc_high_conf = df_probs_with[high_conf_with]['correcto'].mean()
        print(f"\nCon Sentimiento - Probabilidad >= {threshold} o <= {1-threshold}:")
        print(f"  Casos: {high_conf_with.sum()} ({high_conf_with.sum()/len(df_probs_with)*100:.1f}%)")
        print(f"  Accuracy en estos casos: {acc_high_conf:.4f}")


In [ ]:
# Calcular matrices de confusión
cm_with = confusion_matrix(results_with['y_true'], results_with['y_pred'])
cm_without = confusion_matrix(results_without['y_true'], results_without['y_pred'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz con sentimiento
sns.heatmap(cm_with, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Baja', 'Sube'], yticklabels=['Baja', 'Sube'])
axes[0].set_title('Con Sentimiento', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Real')
axes[0].set_xlabel('Predicho')

# Matriz sin sentimiento
sns.heatmap(cm_without, annot=True, fmt='d', cmap='Reds', ax=axes[1],
            xticklabels=['Baja', 'Sube'], yticklabels=['Baja', 'Sube'])
axes[1].set_title('Sin Sentimiento', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Real')
axes[1].set_xlabel('Predicho')

plt.tight_layout()
plt.show()

# Interpretación
print("\nINTERPRETACION DE MATRICES DE CONFUSION")
print("="*60)
print("\nCon Sentimiento:")
print(f"  Verdaderos Negativos (TN): {cm_with[0,0]} - Predijo baja y bajo")
print(f"  Falsos Positivos (FP): {cm_with[0,1]} - Predijo sube pero bajo")
print(f"  Falsos Negativos (FN): {cm_with[1,0]} - Predijo baja pero subio")
print(f"  Verdaderos Positivos (TP): {cm_with[1,1]} - Predijo sube y subio")

print("\nSin Sentimiento:")
print(f"  Verdaderos Negativos (TN): {cm_without[0,0]}")
print(f"  Falsos Positivos (FP): {cm_without[0,1]}")
print(f"  Falsos Negativos (FN): {cm_without[1,0]}")
print(f"  Verdaderos Positivos (TP): {cm_without[1,1]}")


## 11. Curvas de Pérdida durante Entrenamiento


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(results_with['train_losses'], label='Con Sentimiento', linewidth=2, color='#3498db')
ax.plot(results_without['train_losses'], label='Sin Sentimiento', linewidth=2, color='#e74c3c')
ax.set_xlabel('Época', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Curva de Pérdida durante Entrenamiento', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nANALISIS DE CONVERGENCIA")
print("="*60)
print(f"\nCon Sentimiento:")
print(f"  Loss inicial: {results_with['train_losses'][0]:.4f}")
print(f"  Loss final: {results_with['train_losses'][-1]:.4f}")
print(f"  Reduccion: {(1 - results_with['train_losses'][-1]/results_with['train_losses'][0])*100:.2f}%")

print(f"\nSin Sentimiento:")
print(f"  Loss inicial: {results_without['train_losses'][0]:.4f}")
print(f"  Loss final: {results_without['train_losses'][-1]:.4f}")
print(f"  Reduccion: {(1 - results_without['train_losses'][-1]/results_without['train_losses'][0])*100:.2f}%")


## 12. Análisis de Hiperparámetros (Grid Search Simplificado)


In [ ]:
# Cargar resultados de Optuna si existen
optuna_trials_path = Path("src/modelo/data/optuna_trials.csv")
best_params_path = Path("src/modelo/data/best_params.csv")

if optuna_trials_path.exists():
    df_optuna = pd.read_csv(optuna_trials_path)
    print("RESULTADOS DE OPTUNA")
    print("="*60)
    print(f"Total de trials: {len(df_optuna)}")
    
    if 'value' in df_optuna.columns:
        print(f"Mejor AUC: {df_optuna['value'].max():.4f}")
        display(df_optuna.sort_values('value', ascending=False).head(10))
    
    if best_params_path.exists():
        best_params = pd.read_csv(best_params_path)
        print("\nMEJORES PARAMETROS")
        print("="*60)
        display(best_params)
else:
    print("No se encontraron resultados de Optuna.")
    print("Ejecuta src/modelo/optuna_search.py para optimizar hiperparametros")


In [ ]:
# Crear tabla completa con todas las probabilidades
print("TABLA COMPLETA DE PROBABILIDADES Y PREDICCIONES")
print("="*60)
print("\nModelo CON Sentimiento:")
print(f"Total de predicciones: {len(df_probs_with)}")

# Agregar interpretación
df_probs_with['interpretacion'] = df_probs_with.apply(
    lambda row: f"{'OK' if row['correcto'] else 'ERR'} {'Subira' if row['probabilidad'] >= 0.5 else 'Bajara'} ({row['probabilidad']:.1%} confianza) | Real: {'Subio' if row['y_true'] == 1 else 'Bajo'}",
    axis=1
)

display(df_probs_with[['y_true', 'y_pred', 'probabilidad', 'correcto', 'interpretacion']])

# Estadísticas por rango de probabilidad
print("\nANALISIS POR RANGO DE PROBABILIDAD")
print("="*60)

ranges = [
    (0.0, 0.3, "Muy Baja (0-30%)"),
    (0.3, 0.5, "Baja (30-50%)"),
    (0.5, 0.7, "Alta (50-70%)"),
    (0.7, 1.0, "Muy Alta (70-100%)")
]

for min_prob, max_prob, label in ranges:
    mask = (df_probs_with['probabilidad'] >= min_prob) & (df_probs_with['probabilidad'] < max_prob)
    if mask.sum() > 0:
        subset = df_probs_with[mask]
        accuracy = subset['correcto'].mean()
        print(f"\n{label}:")
        print(f"  Casos: {mask.sum()} ({mask.sum()/len(df_probs_with)*100:.1f}%)")
        print(f"  Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)")
        print(f"  Probabilidad promedio: {subset['probabilidad'].mean():.3f}")


## 14. Interpretación de Resultados y Conclusiones


In [ ]:
print("RESUMEN Y CONCLUSIONES")
print("="*60)

print("\n1. COMPARACION DE MODELOS:")
print(f"   - Modelo con sentimiento tiene {len(feature_cols_with)} features")
print(f"   - Modelo sin sentimiento tiene {len(feature_cols_no_sentiment)} features")
print(f"   - Mejora en Accuracy: {df_comparison.iloc[0]['Mejora']:.4f} ({df_comparison.iloc[0]['Mejora %']:.2f}%)")
print(f"   - Mejora en F1: {df_comparison.iloc[3]['Mejora']:.4f} ({df_comparison.iloc[3]['Mejora %']:.2f}%)")

print("\n2. INTERPRETACION DE PROBABILIDADES:")
print("   - Probabilidad = confianza del modelo (0-1)")
print("   - Probabilidad >= 0.5 -> Predice 'subira'")
print("   - Probabilidad < 0.5 -> Predice 'bajara'")
print(f"   - Probabilidad promedio (con sentimiento): {df_probs_with['probabilidad'].mean():.3f}")
print(f"   - Probabilidad promedio (sin sentimiento): {df_probs_without['probabilidad'].mean():.3f}")
print(f"   - Desviacion estandar (con sentimiento): {df_probs_with['probabilidad'].std():.3f}")
print(f"   - Desviacion estandar (sin sentimiento): {df_probs_without['probabilidad'].std():.3f}")

print("\n3. INTERPRETACION DE ACCURACY:")
print(f"   - Accuracy (con sentimiento): {results_with['metrics']['accuracy']:.2%}")
print(f"   - Accuracy (sin sentimiento): {results_without['metrics']['accuracy']:.2%}")
if results_with['metrics']['accuracy'] > 0.5:
    print("   - El modelo es mejor que lanzar una moneda (50%)")
if results_with['metrics']['accuracy'] > 0.6:
    print("   - El modelo tiene buen rendimiento (>60%)")

print("\n4. RECOMENDACIONES:")
if df_comparison.iloc[0]['Mejora'] > 0:
    print("   - Las features de sentimiento mejoran el modelo")
    print("   - Continuar usando sentimiento en el pipeline")
else:
    print("   - Las features de sentimiento no mejoran significativamente")
    print("   - Considerar: mas datos, mejor modelo de sentimiento, o diferentes features")

print("\n5. PROXIMOS PASOS:")
print("   - Recolectar mas datos para mejorar el modelo")
print("   - Probar diferentes arquitecturas (GRU, Transformer)")
print("   - Fine-tuning del modelo de sentimiento")
print("   - Agregar mas features (indicadores tecnicos, volumen)")
print("   - Usar las probabilidades para decisiones de trading (solo si probabilidad > 0.7)")
